<a href="https://colab.research.google.com/github/guidias-sketch/Algebra-linear-multiplicacao-de-matrizes-densas-GEMM/blob/main/%C3%81lgebra_linear_%E2%80%94_multiplica%C3%A7%C3%A3o_de_matrizes_densas_(GEMM).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash

cat << EOF > matrix_multiply.cpp
void cpu_sequential(float* A, float* B, float* C, int N) {
    for (int i = 0; i < N; ++i) {
        for (int j = 0; j < N; ++j) {
            float sum = 0.0f;
            for (int k = 0; k < N; ++k) {
                sum += A[i * N + k] * B[k * N + j];
            }
            C[i * N + j] = sum;
        }
    }
}
EOF

g++ -c matrix_multiply.cpp -o matrix_multiply.o

ls -l matrix_multiply.o

-rw-r--r-- 1 root root 1480 May 26 20:46 matrix_multiply.o


In [15]:
%%writefile cuda_naive.cu

__global__ void cuda_naive(float* A, float* B, float* C, int N) {
    // Calculando a linha e a coluna globais que esta thread vai processar
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // Verificação de limites caso N não seja múltiplo do tamanho do bloco
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; ++k) {
            // Leituras da memória global
            sum += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

Overwriting cuda_naive.cu


In [16]:
%%writefile cuda_tiled.cu
#define TILE_WIDTH 32 // Pode ser usada com 16 ou 32

__global__ void gemm_cuda_tiled(float* A, float* B, float* C, int N) {
    // Alocação da memória compartilhada para os tiles
    __shared__ float sA[TILE_WIDTH][TILE_WIDTH];
    __shared__ float sB[TILE_WIDTH][TILE_WIDTH];

    int tx = threadIdx.x; int ty = threadIdx.y;
    int row = blockIdx.y * TILE_WIDTH + ty;
    int col = blockIdx.x * TILE_WIDTH + tx;

    float sum = 0.0f;

    // Calcula o número de tiles necessários
    int numTiles = (N + TILE_WIDTH - 1) / TILE_WIDTH;

    for (int ph = 0; ph < numTiles; ++ph) {
        // Cada thread carrega um elemento para sA e sB
        if (row < N && (ph * TILE_WIDTH + tx) < N)
            sA[ty][tx] = A[row * N + ph * TILE_WIDTH + tx];
        else
            sA[ty][tx] = 0.0f; // Padding com zero para matrizes irregulares

        if (col < N && (ph * TILE_WIDTH + ty) < N)
            sB[ty][tx] = B[(ph * TILE_WIDTH + ty) * N + col];
        else
            sB[ty][tx] = 0.0f;

        // Aguarda todas as threads do bloco terminarem de carregar os dados
        __syncthreads();

        // Multiplica o tile na memória
        for (int k = 0; k < TILE_WIDTH; ++k) {
            sum += sA[ty][k] * sB[k][tx];
        }

        // Aguarda o cálculo terminar antes da próxima iteração sobrescrever os tiles
        __syncthreads();
    }

    if (row < N && col < N) {
        C[row * N + col] = sum;
    }
}

Overwriting cuda_tiled.cu


In [ ]:
%%bash
nvcc -c cuda_tiled.cu -o cuda_tiled.o
ls -l cuda_tiled.o

-rw-r--r-- 1 root root 10984 May 26 20:46 cuda_tiled.o


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
%%writefile main.cu
#include <iostream>
#include <cuda_runtime.h>

// Declaração correta de kernels definidos em outros arquivos .cu
__global__ void cuda_naive(float* A, float* B, float* C, int N);
__global__ void gemm_cuda_tiled(float* A, float* B, float* C, int N);

extern "C" void run_benchmarks(int N, float* h_A, float* h_B, float* h_C_naive, float* h_C_tiled) {
    float *d_A, *d_B, *d_C;
    size_t size = N * N * sizeof(float);

    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // Configuração para Naive (Blocos 32x32)
    dim3 threads(32, 32);
    dim3 blocks((N + 31) / 32, (N + 31) / 32);

    // Executa Naive
    cuda_naive<<<blocks, threads>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();
    cudaMemcpy(h_C_naive, d_C, size, cudaMemcpyDeviceToHost);

    // Limpa C para o próximo teste
    cudaMemset(d_C, 0, size);

    // Executa Tiled (Mesma configuração de blocos, mas o kernel usa Shared Memory)
    gemm_cuda_tiled<<<blocks, threads>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();
    cudaMemcpy(h_C_tiled, d_C, size, cudaMemcpyDeviceToHost);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
}

Overwriting main.cu


In [ ]:
%%bash
# Recompilando a biblioteca com as correções
nvcc -Xcompiler -fPIC -shared -o libgemm.so main.cu cuda_naive.cu cuda_tiled.cu
ls -l libgemm.so

-rwxr-xr-x 1 root root 1013680 May 26 20:49 libgemm.so


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [17]:
import numpy as np
import ctypes
import time

# Recarrega a biblioteca (forçando o carregamento do novo .so)
try:
    lib = ctypes.CDLL('./libgemm.so')
except OSError as e:
    print(f"Erro ao carregar biblioteca: {e}")

lib.run_benchmarks.argtypes = [ctypes.c_int,
                               ctypes.POINTER(ctypes.c_float),
                               ctypes.POINTER(ctypes.c_float),
                               ctypes.POINTER(ctypes.c_float),
                               ctypes.POINTER(ctypes.c_float)]

def benchmark_gemm(N=1024):
    print(f"Iniciando benchmark para matriz {N}x{N}...")
    A = np.random.rand(N, N).astype(np.float32)
    B = np.random.rand(N, N).astype(np.float32)
    C_naive = np.zeros((N, N), dtype=np.float32)
    C_tiled = np.zeros((N, N), dtype=np.float32)

    # NumPy (Referência usando cuBLAS internamente se disponível)
    start = time.time()
    C_numpy = np.matmul(A, B)
    end = time.time()
    t_numpy = end - start
    print(f"NumPy/Reference: {t_numpy:.4f}s")

    # Preparando ponteiros para o C
    ptr_A = A.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    ptr_B = B.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    ptr_C_naive = C_naive.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    ptr_C_tiled = C_tiled.ctypes.data_as(ctypes.POINTER(ctypes.c_float))

    # Executa kernels CUDA
    start = time.time()
    lib.run_benchmarks(N, ptr_A, ptr_B, ptr_C_naive, ptr_C_tiled)
    end = time.time()
    t_cuda = end - start
    print(f"CUDA Kernels (Tempo total em C++): {t_cuda:.4f}s")

    # Verificação de integridade
    np.testing.assert_allclose(C_numpy, C_tiled, atol=1e-2)
    print("\nSucesso: O resultado do Tiled CUDA coincide com o NumPy!")

    if t_numpy > 0:
        print(f"Relação de velocidade (NumPy vs CUDA Tiled): Aproximadamente {t_numpy/t_cuda:.2f}x")

benchmark_gemm(1024)

Iniciando benchmark para matriz 1024x1024...
NumPy/Reference: 0.0198s
CUDA Kernels (Tempo total em C++): 0.0170s

Sucesso: O resultado do Tiled CUDA coincide com o NumPy!
Relação de velocidade (NumPy vs CUDA Tiled): Aproximadamente 1.16x


In [ ]:
%%bash
# Compila os kernels e a ponte em uma biblioteca compartilhada
nvcc -Xcompiler -fPIC -shared -o libgemm.so main.cu cuda_naive.cu cuda_tiled.cu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [18]:
import numpy as np
import ctypes
import time

# Carrega a biblioteca
lib = ctypes.CDLL('./libgemm.so')
lib.run_benchmarks.argtypes = [ctypes.c_int,
                               ctypes.POINTER(ctypes.c_float),
                               ctypes.POINTER(ctypes.c_float),
                               ctypes.POINTER(ctypes.c_float),
                               ctypes.POINTER(ctypes.c_float)]

def benchmark_gemm(N=1024):
    A = np.random.rand(N, N).astype(np.float32)
    B = np.random.rand(N, N).astype(np.float32)
    C_naive = np.zeros((N, N), dtype=np.float32)
    C_tiled = np.zeros((N, N), dtype=np.float32)

    # NumPy (Referência)
    start = time.time()
    C_numpy = np.matmul(A, B)
    end = time.time()
    print(f"NumPy (CPU/cuBLAS): {end - start:.4f}s")

    # CUDA Kernels
    ptr_A = A.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    ptr_B = B.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    ptr_C_naive = C_naive.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
    ptr_C_tiled = C_tiled.ctypes.data_as(ctypes.POINTER(ctypes.c_float))

    start = time.time()
    lib.run_benchmarks(N, ptr_A, ptr_B, ptr_C_naive, ptr_C_tiled)
    end = time.time()
    print(f"CUDA Kernels (Naive + Tiled total): {end - start:.4f}s")

    # Verificação
    np.testing.assert_allclose(C_numpy, C_tiled, atol=1e-3)
    print("Sucesso: O resultado do Tiled CUDA coincide com o NumPy!")

benchmark_gemm(1024)

NumPy (CPU/cuBLAS): 0.0190s
CUDA Kernels (Naive + Tiled total): 0.0170s
Sucesso: O resultado do Tiled CUDA coincide com o NumPy!
